In [0]:
%run /Shared/gskglobal-ict/bundle_ict/Utilities/Audit_Utils

In [0]:
%python
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "ent_gf_dev")
dbutils.widgets.text("entity_id", "zipt_markup")

catalog = dbutils.widgets.get("catalog")
entity_id = dbutils.widgets.get("entity_id")
AUDIT_TASK_TABLE = f"{catalog}.ict_audit.audit_task_run"
WATERMARK_TABLE = f"{catalog}.ict_metadata.etl_watermark"

cfg = (
    spark.table(f"{catalog}.ict_metadata.etl_setup_config")
    .filter((F.col("entity_id") == entity_id) & (F.col("target_layer") == "unified") & F.col("is_active"))
    .first()
)
if cfg is None:
    raise ValueError(f"No active unified config found for entity_id={entity_id}")

source_table = f"{catalog}.{cfg['source_read_from']}"   # ent_gf_dev.ict_trusted.zipt_markup
target_table = f"{catalog}.{cfg['target_table']}"       # ent_gf_dev.ict_unified.zipt_markup
primary_keys = cfg["primary_key_cols"]   # trusted (renamed) column names
partition_column = cfg["partition_column"]
load_type = cfg["load_type"].upper()

In [0]:
%python
import uuid
from datetime import datetime, UTC
from delta.tables import DeltaTable
from pyspark.sql import Window

run_id = str(uuid.uuid4())
source_schema = source_table.rsplit(".", 1)[0]
target_schema = target_table.rsplit(".", 1)[0]

# Watermark: empty last_load_type = initial load; last_watermark_value = commit timestamp of last processed trusted version
wm = spark.table(WATERMARK_TABLE).filter((F.col("entity_id") == entity_id) & (F.col("layer") == "unified")).first()
is_initial = not wm["last_load_type"]

audit_args = dict(
    table_name=entity_id, run_id=run_id, layer="unified",
    target_table=target_table, load_type=load_type, source_schema=source_schema, start_ts=datetime.now(UTC),
    watermark_from=wm["last_watermark_value"],
)

try:
    # Upper bound = latest trusted version at the start of this run
    history = DeltaTable.forName(spark, source_table).history().select("version", "timestamp")
    end_version, end_ts = history.orderBy(F.col("version").desc()).first()

    if is_initial:
        start_version = 0
    else:
        last_version = history.filter(F.col("timestamp") <= F.lit(wm["last_watermark_value"])).agg(F.max("version")).first()[0]
        if last_version is None:
            raise ValueError(f"Watermark {wm['last_watermark_value']} no longer in trusted history; reset unified watermark for a full reload")
        start_version = last_version + 1

    if start_version > end_version:
        # No new trusted commits: skip merge, watermark unchanged
        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=0,
            message=f"No new commits in {source_table} after version {end_version}",
        ), AUDIT_TASK_TABLE)
        print(f"No new trusted commits after version {end_version}, load skipped")
    else:
        if is_initial:
            # Initial: full trusted snapshot
            window_df = spark.read.option("versionAsOf", end_version).table(source_table)
        else:
            # Incremental: CDF inserts/updates in start_version..end_version, latest change per key
            latest = Window.partitionBy(*primary_keys).orderBy(F.col("_commit_version").desc())
            window_df = (
                spark.read.option("readChangeFeed", "true")
                .option("startingVersion", start_version).option("endingVersion", end_version)
                .table(source_table)
                .filter(F.col("_change_type").isin("insert", "update_postimage"))
                .withColumn("_rn", F.row_number().over(latest))
                .filter("_rn = 1")
                .select(*spark.table(source_table).columns)
            )
        print(f"{'initial' if is_initial else 'incremental'} load from {source_table}: versions {start_version}..{end_version}")

        source_df = (
            window_df.withColumn("_ingest_ts", F.current_timestamp())
            .withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
        )

        if not spark.catalog.tableExists(target_table):
            # First run: write directly
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
            writer = source_df.write.format("delta")
            if partition_column:
                writer = writer.partitionBy(partition_column)
            writer.saveAsTable(target_table)
            rows_written = DeltaTable.forName(spark, target_table).history(1).first()["operationMetrics"]["numOutputRows"]
            rows_read, rows_inserted, rows_updated = rows_written, rows_written, 0
        else:
            # Merge on primary keys
            target = DeltaTable.forName(spark, target_table)
            (
                target.alias("t")
                .merge(source_df.alias("s"), " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in primary_keys))
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            metrics = target.history(1).first()["operationMetrics"]
            rows_read, rows_inserted, rows_updated = (
                metrics["numSourceRows"], metrics["numTargetRowsInserted"], metrics["numTargetRowsUpdated"]
            )

        # Move watermark to commit timestamp of end_version
        spark.sql(
            f"""UPDATE {WATERMARK_TABLE}
                SET previous_watermark_value = last_watermark_value,
                    last_watermark_value = CAST(:end_ts AS TIMESTAMP),
                    last_run_id = :last_run_id,
                    last_load_type = :last_load_type,
                    last_success_ts = current_timestamp(),
                    updated_by = current_user()
                WHERE entity_id = :entity_id AND layer = 'unified'""",
            args={"end_ts": end_ts, "last_run_id": stable_long_id(run_id),
                  "last_load_type": "initial" if is_initial else "incremental", "entity_id": entity_id},
        )

        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), watermark_to=end_ts,
            row_count=rows_read, rows_inserted=rows_inserted, rows_updated=rows_updated,
        ), AUDIT_TASK_TABLE)
        print(f"Loaded {source_table} -> {target_table}: {rows_inserted} inserted, {rows_updated} updated, trusted version {end_version}")

except Exception as e:
    write_task_audit(spark, build_task_audit_details(
        **audit_args, status="FAILED", end_ts=datetime.now(UTC),
        message=str(e)[:2000], rows_updated=None, rows_deleted=None,
    ), AUDIT_TASK_TABLE)
    raise